In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib


In [11]:
DATA_PATH = "../data/processed/flights_ml.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


(196773, 10)


,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,ROUTE,DAY_OF_WEEK,DEP_HOUR,IS_PEAK_HOUR,IS_WEEKEND,DISTANCE,IS_DELAYED
0,OTHER,HSV,IAH,OTHER,1,6,0,0,595.0,0
1,UA,SFO,LAX,SFO_LAX,1,20,0,0,337.0,0
2,AA,CLT,ORD,OTHER,2,11,0,0,599.0,0
3,AA,AUS,LAX,OTHER,4,18,1,0,1242.0,0
4,DL,ATL,SAT,OTHER,1,11,0,0,874.0,0


In [12]:
X = df.drop(columns=["IS_DELAYED"])
y = df["IS_DELAYED"]

print(X.columns)


Index(['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'ROUTE',
       'DAY_OF_WEEK', 'DEP_HOUR', 'IS_PEAK_HOUR', 'IS_WEEKEND', 'DISTANCE'],
      dtype='object')


In [13]:
categorical_features = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "ROUTE"
]

numerical_features = [
    "DAY_OF_WEEK",
    "DEP_HOUR",
    "IS_PEAK_HOUR",
    "IS_WEEKEND",
    "DISTANCE"
]


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)


rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [16]:
lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

lr_model.fit(X_train, y_train)

y_lr_proba = lr_model.predict_proba(X_test)[:, 1]
print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, y_lr_proba))


Logistic Regression ROC-AUC: 0.6571858794689283


In [17]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=18,
        min_samples_split=10,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

y_rf_proba = rf_model.predict_proba(X_test)[:, 1]
print("Random Forest ROC-AUC:", roc_auc_score(y_test, y_rf_proba))


Random Forest ROC-AUC: 0.665624484867905


In [ ]:
param_grid = {
    "classifier__n_estimators": [200, 300],
    "classifier__max_depth": [14, 18, 22],
    "classifier__min_samples_split": [5, 10]
}

grid = GridSearchCV(
    rf_model,
    param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)


Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [9]:
best_model = grid.best_estimator_

y_best_proba = best_model.predict_proba(X_test)[:, 1]
y_best_pred = best_model.predict(X_test)

print("BEST ROC-AUC:", roc_auc_score(y_test, y_best_proba))
print(classification_report(y_test, y_best_pred))
confusion_matrix(y_test, y_best_pred)


NameError: name 'grid' is not defined

In [ ]:
joblib.dump(best_model, "../models/best_model.joblib")
print("Saved model to models/best_model.joblib")


              precision    recall  f1-score   support

           0       0.89      0.59      0.71     32334
           1       0.25      0.65      0.37      7021

    accuracy                           0.60     39355
   macro avg       0.57      0.62      0.54     39355
weighted avg       0.77      0.60      0.64     39355

RF ROC AUC: 0.6638802301399312
